# 04 – Sanasto ja sanamäärät

**Tehtävä:** siivotuista puheista (muistio 02) muodostetaan piirteet (*features*) eli **sanat ja sanaparit**. Mukaan otetaan vain riittävän yleiset, ja jokaiselle puheelle lasketaan niiden esiintymiskerrat. Tuloksena on harva matriisi (*sparse matrix*; puheet × fraasit), jota muistiot 05 (luokittelija) ja 06 (leave-out-mittari) käyttävät.

**Syöte:** `Data/processed/speeches_clean.parquet`
**Tulokset:**
- `Data/processed/vocab.txt`: yksi fraasi per rivi
- `Data/processed/counts.npz`: harva matriisi, analyysiotoksen puheet × fraasit
- `Data/processed/counts_index.parquet`: matriisin rivien taustatiedot samassa järjestyksessä

**Muisti:** sanaston laskeminen vaatii noin 5 Gt muistia, koska kaikki ehdokasfraasit (yli miljoona) lasketaan ensin. Valmis sanasto luetaan tiedostoista (asetus `UUDELLEEN`).

## Miksi sanaparit?

Simola ym. (s. 10) ja Gentzkow ym. (s. 1311) käyttävät **sanapareja** (*bigrams*), koska pari kantaa enemmän merkitystä kuin yksittäinen sana. Simolan ym. esimerkissä *ystävyys.yhteistyö* viittaa YYA-sopimukseen, kun sanat *ystävyys* ja *yhteistyö* erikseen eivät kerro paljon. Suomen yhdyssanat kantavat kuitenkin paljon merkitystä yksinäänkin (*maahanmuuttopolitiikka*, *sosiaaliturvaleikkaus*). Siksi:
- **luokittelija** (muistio 05) käyttää **sanoja ja sanapareja**
- **leave-out-mittari** (muistio 06) käyttää **vain sanapareja** kuten artikkeleissa

Sanaparin sanat yhdistetään alaviivalla, esim. `ulkomaalais_maanomistus`. Sanapareja ei muodosteta katkomerkin `|` yli eli poistetun nimen, puolueen tai muodollisuuden kohdalla.

In [1]:
import os

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer

SIIVOTUT = "../Data/processed/speeches_clean.parquet"
KANSIO = "../Data/processed/"
KATKO = "|"
UUDELLEEN = False      # True: laske sanasto uudelleen, vaikka se on jo tallennettu (vaatii noin 5 Gt muistia)

sarakkeet = ["speech_id", "start_time", "session_year", "term", "person_id", "mp_name", "party",
             "in_government", "is_minister", "speech_type", "n_tokens", "tokens", "in_sample"]
puheet = pd.read_parquet(SIIVOTUT, columns=sarakkeet)
puheet = puheet[puheet["in_sample"]].reset_index(drop=True)
print("Puheita analyysiotoksessa:", len(puheet))

Puheita analyysiotoksessa: 130965


In [2]:
def sanat_ja_parit(teksti):
    """Palauttaa puheen sanat ja sanaparit. Pareja ei muodosteta katkomerkin '|' yli."""
    tulos = []
    for jakso in teksti.split(KATKO):
        sanat = jakso.split()
        tulos.extend(sanat)
        for i in range(len(sanat) - 1):
            tulos.append(sanat[i] + "_" + sanat[i + 1])
    return tulos


print(sanat_ja_parit("hallitus leik | linj väärä"))

['hallitus', 'leik', 'hallitus_leik', 'linj', 'väärä', 'linj_väärä']


## Ehdokasfraasit

Lasketaan kaikki sanat ja sanaparit kaikista puheista. `min_df=3` karsii jo tässä fraasit, jotka esiintyvät alle kolmessa puheessa; ne eivät voisi läpäistä alla olevaa ehtoa 3.

## Esiintymisrajat (Simola ym. s. 10; Gentzkow ym. s. 1312)

Fraasi otetaan sanastoon, jos
1. sitä käytetään **vähintään 100 kertaa** koko aineistossa,
2. sitä käytetään **vähintään 10 kertaa jonakin valtiopäivävuonna** ja
3. sitä käyttää **vähintään 10 puhuja-vuotta** (sama edustaja eri vuonna lasketaan erikseen).

Rajat poistavat harvinaiset fraasit, jotka näyttäisivät puoluesidonnaisilta sattumalta, sekä harvojen puhujien maneerit, ja ne pienentävät matriisia. Sanasto on sama kaikille kausille, joten kaudet ovat vertailukelpoisia. Haittapuoli (Simola ym. s. 11): aineiston lopussa yleistyneet fraasit ehtivät kertyä vähemmän, mikä koskee erityisesti keskeneräistä kautta 2023–27.

Tulokset tallennetaan kansioon `Data/processed/`. Jos ne ovat jo olemassa, muistio lukee ne eikä laske sanastoa uudelleen (asetus `UUDELLEEN`).

In [3]:
def summat_ryhmittain(avain, X):
    """Laskee X:n rivit (puheet) yhteen avaimen mukaan: tulos on ryhmät × fraasit."""
    _, ryhma = np.unique(avain, return_inverse=True)
    yhdistys = sp.csr_matrix((np.ones(len(ryhma)), (ryhma, np.arange(len(ryhma)))))
    return sp.csr_matrix(yhdistys @ X)


VALMIIT = [KANSIO + nimi for nimi in ["vocab.txt", "counts.npz", "counts_index.parquet"]]

if UUDELLEEN or not all(os.path.exists(t) for t in VALMIIT):
    laskija = CountVectorizer(analyzer=sanat_ja_parit, min_df=3, dtype=np.int32)
    X = sp.csc_matrix(laskija.fit_transform(puheet["tokens"]))
    fraasit = np.array(laskija.get_feature_names_out())
    print("Ehdokasfraaseja:", len(fraasit))

    vuodet = puheet["session_year"].astype(str)
    yhteensa = np.asarray(X.sum(axis=0)).ravel()                                          # ehto 1
    suurin_vuodessa = summat_ryhmittain(vuodet, X).max(axis=0).toarray().ravel()           # ehto 2
    puhuja_vuosi = puheet["person_id"].astype(str) + "_" + vuodet
    puhuja_vuosia = np.asarray((summat_ryhmittain(puhuja_vuosi, X) > 0).sum(axis=0)).ravel()  # ehto 3

    ehto1 = yhteensa >= 100
    ehto2 = suurin_vuodessa >= 10
    ehto3 = puhuja_vuosia >= 10
    print("Ehto 1 (≥100 kertaa):", ehto1.sum())
    print("Ehdot 1 ja 2 (≥10 kertaa jonakin vuonna):", (ehto1 & ehto2).sum())
    print("Kaikki ehdot (≥10 puhuja-vuotta):", (ehto1 & ehto2 & ehto3).sum())

    mukaan = ehto1 & ehto2 & ehto3
    sanasto = fraasit[mukaan]
    X = X[:, mukaan].tocsr()

    with open(KANSIO + "vocab.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(sanasto) + "\n")
    sp.save_npz(KANSIO + "counts.npz", X)
    puheet.drop(columns="tokens").to_parquet(KANSIO + "counts_index.parquet", index=False)
    print("Tallennettu: vocab.txt, counts.npz, counts_index.parquet")
else:
    print("Sanasto on jo laskettu, luetaan tiedostoista.")
    with open(KANSIO + "vocab.txt", encoding="utf-8") as f:
        sanasto = np.array(f.read().split())
    X = sp.load_npz(KANSIO + "counts.npz").tocsr()

on_pari = np.array(["_" in f for f in sanasto])
print("Sanastossa", len(sanasto), "fraasia:", (~on_pari).sum(), "sanaa ja", on_pari.sum(), "sanaparia")
print("(Simolalla ym. noin 53 000 sanaparia 110 vuoden aineistosta)")

Sanasto on jo laskettu, luetaan tiedostoista.


Sanastossa 26209 fraasia: 13468 sanaa ja 12741 sanaparia
(Simolalla ym. noin 53 000 sanaparia 110 vuoden aineistosta)


### Millaisia fraaseja sanastossa on?

Yleisimmät sanaparit ja satunnainen otos. Sanat ovat vartaloita, joten ne näyttävät katkaistuilta.

In [4]:
kaytto = np.asarray(X.sum(axis=0)).ravel()
parit = pd.DataFrame({"fraasi": sanasto[on_pari], "kertoja": kaytto[on_pari]})
print("Yleisimmät sanaparit:")
print(parit.sort_values("kertoja", ascending=False).head(25).to_string(index=False))
print()
print("Satunnainen otos sanapareja:")
print(parit.sample(20, random_state=1).to_string(index=False))

Yleisimmät sanaparit:
           fraasi  kertoja
         täl_hetk    28928
        muun_muas    21900
     miljoon_euro    21516
hallituks_esityks    13246
     miljard_euro     8889
        sama_aika     8794
          tot_kai     7607
          en_kaik     7492
 hallituks_esitys     6746
          nost_es     6296
     euroop_union     6289
    julkis_taloud     6281
         koko_aja     6263
         en_vuode     6132
        erit_hyvä     5281
        sama_miel     5240
      erit_tärkeä     5057
      viime_kaude     5047
          en_vuon     4982
        tasa_arvo     4944
   kiinnit_huomio     4811
       vuode_aika     4799
         tääl_sal     4211
          hyvä_as     4096
     tääl_eduskun     3915

Satunnainen otos sanapareja:
                fraasi  kertoja
             tärk_myös      319
              voi_kaik      133
             joil_alue      138
           täysi_ikäis      143
           aikais_tääl      129
              tarv_uus      141
     yksittäis_henkilö

In [5]:
fraaseja_per_puhe = np.asarray(X.sum(axis=1)).ravel()
print("Sanaston fraaseja per puhe:")
print(pd.Series(fraaseja_per_puhe).describe(percentiles=[0.1, 0.5, 0.9]).round(0))
print(f"Puheita, joissa alle 20 fraasia: {(fraaseja_per_puhe < 20).mean():.1%} (jätetään luokittelijasta pois)")

Sanaston fraaseja per puhe:
count    130965.0
mean        166.0
std         131.0
min           0.0
10%          64.0
50%         107.0
90%         365.0
max        1900.0
dtype: float64
Puheita, joissa alle 20 fraasia: 0.6% (jätetään luokittelijasta pois)
